# Etapa 4 - Treinamento de Modelos

## Objetivo
Nesta etapa vamos treinar modelos de classificação e comparar o desempenho para prever churn.

## O que vamos fazer?
1. carregar os dados prontos do preprocessamento
2. separar features e alvo
3. treinar modelos iniciais
4. comparar métricas
5. escolher o melhor modelo
6. salvar o modelo final

## Modelos que vamos testar
- Regressão Logística
- Random Forest

## Importante
Em problemas de churn, não basta olhar só acurácia. Também precisamos olhar recall, precisão, F1 e ROC-AUC.

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import joblib

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [27]:
# Definir caminhos do projeto
try:
    notebook_dir = Path.cwd()
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
except:
    project_root = Path.cwd()

processed_dir = project_root / 'data' / 'processed'
artifacts_dir = project_root / 'artifacts'
models_dir = artifacts_dir / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

train_path = processed_dir / '03_X_train_preprocessed.csv'
test_path = processed_dir / '04_X_test_preprocessed.csv'

print(f'Projeto: {project_root}')
print(f'Treino: {train_path}')
print(f'Teste: {test_path}')

Projeto: c:\Temp\telco-churn-ml
Treino: c:\Temp\telco-churn-ml\data\processed\03_X_train_preprocessed.csv
Teste: c:\Temp\telco-churn-ml\data\processed\04_X_test_preprocessed.csv


In [28]:
# Carregar dados de treino e teste
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('Treino shape:', train_df.shape)
print('Teste shape:', test_df.shape)
print('\nPrimeiras 5 linhas do treino:')
print(train_df.head())

Treino shape: (5616, 46)
Teste shape: (1405, 46)

Primeiras 5 linhas do treino:
   Churn         0         1         2         3    4    5    6    7    8  \
0      1 -0.440315 -1.241331  0.193165 -0.946349  0.0  1.0  1.0  0.0  1.0   
1      0 -0.440315 -0.711078  0.647355 -0.435345  1.0  0.0  1.0  0.0  1.0   
2      0 -0.440315  1.409933  0.820380  1.794547  0.0  1.0  0.0  1.0  0.0   
3      0 -0.440315 -1.118965  0.023467 -0.858745  0.0  1.0  1.0  0.0  1.0   
4      0 -0.440315 -0.262403 -1.483847 -0.783388  0.0  1.0  1.0  0.0  1.0   

   ...   35   36   37   38   39   40   41   42   43   44  
0  ...  0.0  1.0  0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  
1  ...  0.0  1.0  0.0  0.0  0.0  1.0  0.0  1.0  0.0  0.0  
2  ...  1.0  1.0  0.0  0.0  0.0  1.0  1.0  0.0  0.0  0.0  
3  ...  1.0  1.0  0.0  0.0  0.0  1.0  1.0  0.0  0.0  0.0  
4  ...  0.0  1.0  0.0  0.0  0.0  1.0  0.0  0.0  1.0  0.0  

[5 rows x 46 columns]


In [29]:
# Separar features e alvo
X_train = train_df.drop(columns=['Churn'])
y_train = train_df['Churn']

X_test = test_df.drop(columns=['Churn'])
y_test = test_df['Churn']

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('\nDistribuição do alvo no treino:')
print(y_train.value_counts())

print('\nDistribuição do alvo no teste:')
print(y_test.value_counts())

X_train shape: (5616, 45)
y_train shape: (5616,)

Distribuição do alvo no treino:
Churn
0    4131
1    1485
Name: count, dtype: int64

Distribuição do alvo no teste:
Churn
0    1033
1     372
Name: count, dtype: int64


In [30]:
# Modelo recomendado para churn: RandomForest com balanceamento e threshold ajustado
models = {
    'RandomForest_recomendado': RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        class_weight='balanced'
    )
}

print('Modelo definido:')
for name in models:
    print(f' - {name}')

Modelo definido:
 - RandomForest_recomendado


In [31]:
# Função para avaliar modelos com limiar de decisão

def evaluate_with_threshold(model, X_train, y_train, X_test, y_test, thresholds=None):
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]

    if thresholds is None:
        thresholds = np.linspace(0.40, 0.70, 7)

    best_result = None

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_prob)
        }

        candidate = {
            'threshold': threshold,
            'metrics': metrics,
            'y_pred': y_pred,
            'y_prob': y_prob
        }

        if best_result is None:
            best_result = candidate
        else:
            current_score = (metrics['f1'], metrics['recall'])
            best_score = (best_result['metrics']['f1'], best_result['metrics']['recall'])
            if current_score > best_score:
                best_result = candidate

    return model, best_result

results = {}
for model_name, model in models.items():
    print('\n' + '=' * 60)
    print(f'Treinando {model_name}')
    print('=' * 60)

    trained_model, best_result = evaluate_with_threshold(
        model,
        X_train,
        y_train,
        X_test,
        y_test,
        thresholds=np.linspace(0.40, 0.70, 7)
    )

    results[model_name] = {
        'model': trained_model,
        'threshold': best_result['threshold'],
        'metrics': best_result['metrics'],
        'y_prob': best_result['y_prob']
    }

    print(f'\nMelhor limiar para {model_name}: {best_result["threshold"]:.2f}')
    for metric_name, value in best_result['metrics'].items():
        print(f'  {metric_name}: {value:.4f}')


Treinando RandomForest_recomendado

Melhor limiar para RandomForest_recomendado: 0.60
  accuracy: 0.7922
  precision: 0.5935
  recall: 0.6828
  f1: 0.6350
  roc_auc: 0.8419


In [32]:
# Comparar resultados com o limiar ajustado
comparison_df = pd.DataFrame(
    {name: info['metrics'] for name, info in results.items()}
).T

comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

print('Comparativo do modelo recomendado:')
print(comparison_df.round(4))

best_model_name = comparison_df['f1'].idxmax()
best_model = results[best_model_name]['model']
best_threshold = results[best_model_name]['threshold']

print(f'\nMelhor modelo pela métrica F1: {best_model_name}')
print(f'Melhor limiar: {best_threshold:.2f}')

Comparativo do modelo recomendado:
                          accuracy  precision  recall     f1  roc_auc
RandomForest_recomendado    0.7922     0.5935  0.6828  0.635   0.8419

Melhor modelo pela métrica F1: RandomForest_recomendado
Melhor limiar: 0.60


In [33]:
# Salvar o melhor modelo junto com o limiar escolhido
model_path = models_dir / 'best_model.pkl'
bundle = {
    'model_name': best_model_name,
    'model': best_model,
    'threshold': best_threshold
}
joblib.dump(bundle, model_path)

print(f'\nModelo salvo em: {model_path}')
print('\nPróximo passo: avaliação final usando o threshold ajustado.')


Modelo salvo em: c:\Temp\telco-churn-ml\artifacts\models\best_model.pkl

Próximo passo: avaliação final usando o threshold ajustado.
